In [0]:
# Read CSV File.
df = spark.read.csv("/Volumes/workspace/default/co2_project",
                    header = True,
                    inferSchema = True)

In [0]:
print(df.columns)

['Make', 'Model', 'Vehicle Class', 'Engine Size(L)', 'Cylinders', 'Transmission', 'Fuel Type', 'Fuel Consumption City (L/100 km)', 'Fuel Consumption Hwy (L/100 km)', 'Fuel Consumption Comb (L/100 km)', 'Fuel Consumption Comb (mpg)', 'CO2 Emissions(g/km)']


In [0]:
# Rename Columns
df = df.withColumnRenamed("Make", "make") \
       .withColumnRenamed("Model", "model") \
       .withColumnRenamed("Vehicle Class", "vehicle_class") \
       .withColumnRenamed("Engine Size(L)", "engine_size") \
       .withColumnRenamed("Cylinders", "cylinders") \
       .withColumnRenamed("Transmission", "transmission") \
       .withColumnRenamed("Fuel Type", "fuel_type") \
       .withColumnRenamed("Fuel Consumption City (L/100 km)", "fuel_city") \
       .withColumnRenamed("Fuel Consumption Hwy (L/100 km)", "fuel_hwy") \
       .withColumnRenamed("Fuel Consumption Comb (L/100 km)", "fuel_comb") \
       .withColumnRenamed("Fuel Consumption Comb (mpg)", "fuel_comb_mpg") \
       .withColumnRenamed("CO2 Emissions(g/km)", "co2_emissions")

print(df.columns)

['make', 'model', 'vehicle_class', 'engine_size', 'cylinders', 'transmission', 'fuel_type', 'fuel_city', 'fuel_hwy', 'fuel_comb', 'fuel_comb_mpg', 'co2_emissions']


In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.default.co2_emissions_raw")

EDA SQL

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.default.co2_emissions_raw;

total_records
7385


In [0]:
%sql
SELECT fuel_type, COUNT(*) AS total
FROM workspace.default.co2_emissions_raw
GROUP BY fuel_type
ORDER BY total DESC;

fuel_type,total
X,3637
Z,3202
E,370
D,175
N,1


In [0]:
%sql
SELECT vehicle_class, AVG(co2_emissions) AS avg_co2
FROM workspace.default.co2_emissions_raw
GROUP BY vehicle_class
ORDER BY avg_co2 DESC;

vehicle_class,avg_co2
VAN - PASSENGER,397.2121212121212
VAN - CARGO,361.5
SUV - STANDARD,304.83673469387753
PICKUP TRUCK - STANDARD,301.51301115241637
PICKUP TRUCK - SMALL,278.9685534591195
TWO-SEATER,277.454347826087
FULL-SIZE,263.31611893583727
MINIVAN,262.3125
SUBCOMPACT,246.44884488448844
STATION WAGON - MID-SIZE,238.69811320754718


In [0]:
%sql
SELECT 
  fuel_type,
  AVG(fuel_city) AS avg_city_fuel,
  AVG(fuel_hwy) AS avg_highway_fuel,
  AVG(co2_emissions) AS avg_co2
FROM workspace.default.co2_emissions_raw
GROUP BY fuel_type;

fuel_type,avg_city_fuel,avg_highway_fuel,avg_co2
Z,13.197532792005031,9.255340412242365,266.0434103685197
D,10.057714285714281,7.3337142857142785,237.54857142857142
X,11.412482815507289,8.459747044267273,235.11932911740445
E,19.429729729729736,13.720000000000004,275.0918918918919
N,15.2,9.5,213.0


In [0]:
%sql
SELECT 
  cylinders,
  AVG(engine_size) AS avg_engine_size,
  AVG(co2_emissions) AS avg_co2
FROM workspace.default.co2_emissions_raw
GROUP BY cylinders
ORDER BY cylinders;

cylinders,avg_engine_size,avg_co2
3,1.3231578947368425,174.89473684210526
4,2.006552795031067,204.85993788819877
5,2.5,235.69230769230768
6,3.36823385118564,263.44685200327064
8,5.164835948644784,320.6191155492154
10,5.619047619047614,361.3095238095238
12,6.2615894039735185,381.0331125827815
16,8.0,522.0


Cleaning Data With Python

In [0]:
df = spark.table("workspace.default.co2_emissions_raw")

df = df.dropna()

df = df.dropDuplicates()

display(df)

make,model,vehicle_class,engine_size,cylinders,transmission,fuel_type,fuel_city,fuel_hwy,fuel_comb,fuel_comb_mpg,co2_emissions
ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221
ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136
ACURA,MDX 4WD,SUV - SMALL,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
ACURA,RDX AWD,SUV - SMALL,3.5,6,AS6,Z,12.1,8.7,10.6,27,244
ACURA,RLX,MID-SIZE,3.5,6,AS6,Z,11.9,7.7,10.0,28,230
ACURA,TL,MID-SIZE,3.5,6,AS6,Z,11.8,8.1,10.1,28,232
ACURA,TL AWD,MID-SIZE,3.7,6,AS6,Z,12.8,9.0,11.1,25,255
ACURA,TL AWD,MID-SIZE,3.7,6,M6,Z,13.4,9.5,11.6,24,267
ACURA,TSX,COMPACT,2.4,4,AS5,Z,10.6,7.5,9.2,31,212


In [0]:
df.printSchema()

root
 |-- make: string (nullable = true)
 |-- model: string (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- engine_size: double (nullable = true)
 |-- cylinders: integer (nullable = true)
 |-- transmission: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- fuel_city: double (nullable = true)
 |-- fuel_hwy: double (nullable = true)
 |-- fuel_comb: double (nullable = true)
 |-- fuel_comb_mpg: integer (nullable = true)
 |-- co2_emissions: integer (nullable = true)



In [0]:
categorical_cols = [
    "make",
    "vehicle_class",
    "transmission",
    "fuel_type"
]

numeric_cols = [
    "engine_size",
    "cylinders",
    "fuel_city",
    "fuel_hwy",
    "fuel_comb",
    "fuel_comb_mpg"
]

label_col = "co2_emissions"

In [0]:
df = df.limit(2000)

In [0]:
# Train-Test Split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.limit(500))
print("Testing rows:", test_df.limit(500))

Training rows: DataFrame[make: string, model: string, vehicle_class: string, engine_size: double, cylinders: int, transmission: string, fuel_type: string, fuel_city: double, fuel_hwy: double, fuel_comb: double, fuel_comb_mpg: int, co2_emissions: int]
Testing rows: DataFrame[make: string, model: string, vehicle_class: string, engine_size: double, cylinders: int, transmission: string, fuel_type: string, fuel_city: double, fuel_hwy: double, fuel_comb: double, fuel_comb_mpg: int, co2_emissions: int]


In [0]:
# Building ML Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml import Pipeline

indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + "_index",
        handleInvalid="keep"
    )
    for col in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=col + "_index",
        outputCol=col + "_encoded"
    )
    for col in categorical_cols
]

feature_cols = [col + "_encoded" for col in categorical_cols] + numeric_cols

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [0]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=label_col,
    numTrees=100,
    maxDepth=8,
    seed=42
)

rf_pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])
rf_model = rf_pipeline.fit(train_df)

rf_predictions = rf_model.transform(test_df)
display(rf_predictions.select("co2_emissions", "prediction"))

co2_emissions,prediction
261,252.47825798159775
235,234.03312023134683
232,234.63516556946183
217,210.5873352207481
327,330.65035545774845
332,330.65035545774845
388,378.28421128257315
390,378.28421128257315
343,342.56725463699325
338,342.56725463699325


In [0]:
gbt = GBTRegressor(
    featuresCol="features",
    labelCol=label_col,
    maxIter=100,
    maxDepth=5,
    seed=42
)

gbt_pipeline = Pipeline(stages=indexers + encoders + [assembler, gbt])
gbt_model = gbt_pipeline.fit(train_df)

gbt_predictions = gbt_model.transform(test_df)
display(gbt_predictions.select("co2_emissions", "prediction"))

co2_emissions,prediction
261,255.7028721561129
235,232.14084287805215
232,233.18878267398244
217,211.275076725653
327,330.2844225084146
332,330.2844225084146
388,378.5037037176136
390,378.5037037176136
343,342.0689575515068
338,342.0689575515068


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_evaluator = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="r2"
)

def evaluate_model(predictions, model_name):
    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)
    
    print(model_name)
    print("RMSE:", rmse)
    print("MAE:", mae)
    print("R2:", r2)
    print("-------------------")

evaluate_model(rf_predictions, "Random Forest")
evaluate_model(gbt_predictions, "Gradient Boosted Tree")

Random Forest
RMSE: 5.397391573287225
MAE: 3.479761589762867
R2: 0.991720494810283
-------------------
Gradient Boosted Tree
RMSE: 4.844885302714705
MAE: 3.4210468552233446
R2: 0.993328806625691
-------------------


In [0]:
final_predictions = gbt_predictions.select(
    "make",
    "model",
    "vehicle_class",
    "engine_size",
    "cylinders",
    "fuel_type",
    "fuel_comb",
    "co2_emissions",
    "prediction"
)

final_predictions.write.mode("overwrite").format("delta").saveAsTable(
    "workspace.default.co2_emission_predictions"
)

In [0]:
%sql
SELECT *
FROM workspace.default.co2_emission_predictions
LIMIT 20;

make,model,vehicle_class,engine_size,cylinders,fuel_type,fuel_comb,co2_emissions,prediction
ACURA,NSX,TWO-SEATER,3.5,6,Z,11.0,261,255.7028721561129
ACURA,RLX,MID-SIZE,3.5,6,Z,10.0,235,232.14084287805215
ACURA,TL,MID-SIZE,3.5,6,Z,10.1,232,233.18878267398244
ALFA ROMEO,GIULIA AWD,MID-SIZE,2.0,4,Z,9.2,217,211.275076725653
ASTON MARTIN,RAPIDE S,SUBCOMPACT,6.0,12,Z,14.2,327,330.2844225084146
ASTON MARTIN,RAPIDE S,SUBCOMPACT,6.0,12,Z,14.2,332,330.2844225084146
ASTON MARTIN,V12 VANTAGE S,TWO-SEATER,6.0,12,Z,16.4,388,378.5037037176136
ASTON MARTIN,V12 VANTAGE S,TWO-SEATER,6.0,12,Z,16.5,390,378.5037037176136
ASTON MARTIN,V8 VANTAGE,TWO-SEATER,4.7,8,Z,14.7,343,342.0689575515068
ASTON MARTIN,V8 VANTAGE S,TWO-SEATER,4.7,8,Z,14.7,338,342.0689575515068


In [0]:
%sql
SELECT 
  vehicle_class,
  AVG(co2_emissions) AS actual_avg_co2,
  AVG(prediction) AS predicted_avg_co2
FROM workspace.default.co2_emission_predictions
GROUP BY vehicle_class;

vehicle_class,actual_avg_co2,predicted_avg_co2
TWO-SEATER,259.58620689655174,258.35969060814216
MID-SIZE,234.2741935483871,234.33428908904574
SUBCOMPACT,252.21212121212122,251.468627560318
COMPACT,208.80392156862746,209.25826252820846
SUV - SMALL,231.19672131147541,230.58778493397762
SUV - STANDARD,311.2352941176471,311.27199240407975
FULL-SIZE,272.5581395348837,272.2419084355856
STATION WAGON - SMALL,180.76923076923077,182.4322837405521
PICKUP TRUCK - SMALL,276.0,276.16575836893526
VAN - CARGO,353.6666666666667,339.946689644895


**Average CO₂ by Vehicle **Class****

In [0]:
%sql
SELECT vehicle_class, AVG(co2_emissions) AS avg_co2
FROM workspace.default.co2_emissions_raw
GROUP BY vehicle_class
ORDER BY avg_co2 DESC;

vehicle_class,avg_co2
VAN - PASSENGER,397.2121212121212
VAN - CARGO,361.5
SUV - STANDARD,304.83673469387753
PICKUP TRUCK - STANDARD,301.51301115241637
PICKUP TRUCK - SMALL,278.9685534591195
TWO-SEATER,277.454347826087
FULL-SIZE,263.31611893583727
MINIVAN,262.3125
SUBCOMPACT,246.44884488448844
STATION WAGON - MID-SIZE,238.69811320754718


Databricks visualization. Run in Databricks to view.

**CO₂ by Fuel Type**

In [0]:
%sql
SELECT 
  fuel_type,
  AVG(co2_emissions) AS avg_co2
FROM co2_emissions_raw
GROUP BY fuel_type
ORDER BY avg_co2 DESC;

fuel_type,avg_co2
E,275.0918918918919
Z,266.0434103685197
D,237.54857142857142
X,235.11932911740445
N,213.0


Databricks visualization. Run in Databricks to view.

**Engine Size vs CO₂**

In [0]:
%sql
SELECT 
  engine_size,
  co2_emissions
FROM co2_emissions_raw;

engine_size,co2_emissions
2.0,196
2.4,221
1.5,136
3.5,255
3.5,244
3.5,230
3.5,232
3.7,255
3.7,267
2.4,212


Databricks visualization. Run in Databricks to view.

**Actual vs Predicted CO₂**

In [0]:
%sql
SELECT 
  co2_emissions,
  prediction
FROM co2_emission_predictions;

co2_emissions,prediction
261,255.7028721561129
235,232.14084287805215
232,233.18878267398244
217,211.275076725653
327,330.2844225084146
332,330.2844225084146
388,378.5037037176136
390,378.5037037176136
343,342.0689575515068
338,342.0689575515068


Databricks visualization. Run in Databricks to view.

**Average Actual vs Predicted by Vehicle Class**

In [0]:
%sql
SELECT 
  vehicle_class,
  AVG(co2_emissions) AS actual_avg_co2,
  AVG(prediction) AS predicted_avg_co2
FROM co2_emission_predictions
GROUP BY vehicle_class
ORDER BY actual_avg_co2 DESC;

vehicle_class,actual_avg_co2,predicted_avg_co2
VAN - PASSENGER,403.2,397.025734470094
VAN - CARGO,353.6666666666667,339.946689644895
SUV - STANDARD,311.2352941176471,311.27199240407975
STATION WAGON - MID-SIZE,306.0,311.5840939811472
PICKUP TRUCK - STANDARD,305.8636363636364,304.98908814536713
PICKUP TRUCK - SMALL,276.0,276.16575836893526
FULL-SIZE,272.5581395348837,272.2419084355856
MINIVAN,272.0,273.3535322440599
TWO-SEATER,259.58620689655174,258.35969060814216
SUBCOMPACT,252.21212121212122,251.468627560318


Databricks visualization. Run in Databricks to view.